# Examen de Recuperación de Información## Ramirez Henry


#### Primero vamos a cargar la libreria nltk y kagglehub la una para la parte del preprocesamiento y la otra para conectarnos con kaggle y cargar los dataset

In [1]:
!pip install nltk kagglehub


   ---------------------------------------- 0.0/68.3 kB ? eta -:--:--
   ------ --------------------------------- 10.2/68.3 kB ? eta -:--:--
   ------------------ --------------------- 30.7/68.3 kB 435.7 kB/s eta 0:00:01
   ------------------ --------------------- 30.7/68.3 kB 435.7 kB/s eta 0:00:01
   ---------------------------------------- 68.3/68.3 kB 411.5 kB/s eta 0:00:00


#### Vamos a cargar las librerias necesarias junto con la descargar de lo necesario para ejecutar ntlk

In [2]:
import os
import re
import math
import pandas as pd
import numpy as np
import kagglehub
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\henry\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\henry\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

##### Conectamos con kaggle y obtenemos la ruta del dataset

In [15]:
import os
import pandas as pd
import kagglehub

# Descargar dataset
path = kagglehub.dataset_download("stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset")
print("Path:", path)



Path: C:\Users\henry\.cache\kagglehub\datasets\stefanoleone992\rotten-tomatoes-movies-and-critic-reviews-dataset\versions\1


#### Cargamos los csv que necesitamos para el sistema

In [17]:
movies_path = os.path.join(path, "rotten_tomatoes_movies.csv")
reviews_path = os.path.join(path, "rotten_tomatoes_critic_reviews.csv")

df_movies = pd.read_csv(movies_path)
df_reviews = pd.read_csv(reviews_path)

df_movies.head()


,rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,...,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Criterion Collection,Certified-Fresh,100.0,54.0,Upright,97.0,105386.0,6,54,0
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Disney,Fresh,89.0,27.0,Upright,74.0,68918.0,5,24,3


#### Aquí obtenemos el nombre de las columnas para luego no tener erores al escribir lo demás

In [19]:
# Ajusta según nombres reales de columnas en tu CSV
# Mostramos las columnas disponibles para elegir las correctas
print("Movies columns:", list(movies.columns))
print("Reviews columns:", list(reviews.columns))


Movies columns: ['rotten_tomatoes_link', 'movie_title', 'movie_info', 'critics_consensus', 'content_rating', 'genres', 'directors', 'authors', 'actors', 'original_release_date', 'streaming_release_date', 'runtime', 'production_company', 'tomatometer_status', 'tomatometer_rating', 'tomatometer_count', 'audience_status', 'audience_rating', 'audience_count', 'tomatometer_top_critics_count', 'tomatometer_fresh_critics_count', 'tomatometer_rotten_critics_count']
Reviews columns: ['rotten_tomatoes_link', 'critic_name', 'top_critic', 'publisher_name', 'review_type', 'review_score', 'review_date', 'review_content']


#### Aquí se realiza: tokenización, minúsculas, stopwords, stemming, limpieza de texto

In [21]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def preprocesar(texto):
    texto = limpiar_texto(texto)
    tokens = nltk.word_tokenize(texto)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens


#### Se realiza la creción del documento relevante con las columnas que nos importan

In [23]:
documentos = []

for _, fila in df_movies.iterrows():
    texto = ""

    if isinstance(fila["movie_title"], str):
        texto += fila["movie_title"] + " "

    if isinstance(fila["movie_info"], str):
        texto += fila["movie_info"] + " "

    if isinstance(fila["genres"], str):
        texto += fila["genres"] + " "

    if isinstance(fila["actors"], str):
        texto += fila["actors"] + " "

    if isinstance(fila["directors"], str):
        texto += fila["directors"] + " "

    documentos.append(texto)

len(documentos)


17712

#### Le aplicamos el preprocesamiento al documento relevante

In [25]:
docs_proc = [preprocesar(doc) for doc in documentos]


#### Creación del vocabulario

In [27]:
vocabulario = sorted(list(set([t for doc in docs_proc for t in doc])))
vocab_index = {t:i for i,t in enumerate(vocabulario)}

print("Tamaño del vocabulario:", len(vocabulario))


Tamaño del vocabulario: 108504


#### Construcción de la matriz TF usando matrices dispersas (En el final explico porque matrices dispersas)

In [69]:
from scipy.sparse import lil_matrix, csr_matrix
import numpy as np

N = len(docs_proc)               
V = len(vocabulario)             

tf = lil_matrix((N, V), dtype=np.float32)

for i, doc in enumerate(docs_proc):
    for token in doc:
        if token in vocab_index:
            tf[i, vocab_index[token]] += 1

tf = tf.tocsr()

print("TF OK:", tf.shape)

TF OK: (17712, 108504)


#### Cálculo de DF e IDF

In [72]:
df = np.diff(tf.tocsc().indptr)  # Frecuencia del documento
idf = np.log((N / (df + 1)))     # IDF


#### Cálculo del TF-IDF

In [75]:
from scipy.sparse import diags

idf_matrix = diags(idf)     
tfidf = tf.dot(idf_matrix) 

print("TF-IDF OK:", tfidf.shape)


TF-IDF OK: (17712, 108504)


#### Ahora definimos la función buscar, que permite realizar consultas textuales sobre el corpus de películas usando TF-IDF. (Igual con vectores dispersos)

In [78]:
from scipy.sparse import csr_matrix
import numpy as np

def buscar(query, k=10):
    tokens = preprocesar(query)

    indices = []
    values = []

    for t in tokens:
        if t in vocab_index:
            idx = vocab_index[t]
            indices.append(idx)
            values.append(1.0)

    if len(indices) == 0:
        return [], []

    q_vec = csr_matrix(
        (values, ([0]*len(indices), indices)),
        shape=(1, len(vocabulario))
    )

    norm_q = np.sqrt(q_vec.multiply(q_vec).sum())
    if norm_q > 0:
        q_vec = q_vec / norm_q

    sim = tfidf.dot(q_vec.T).toarray().ravel()

    top_idx = np.argsort(sim)[::-1][:k]

    return top_idx, sim[top_idx]


#### Esta es la función para buscar va a mostrar el título de la película, la información de la película y el score obtenido

In [40]:
def mostrar_resultados(top_idx, sim):
    print("\nRESULTADOS:")
    for rank, (i, s) in enumerate(zip(top_idx, sim), start=1):
        titulo = df_movies.iloc[i]["movie_title"]
        info = df_movies.iloc[i]["movie_info"]
        print(f"{rank}. {titulo} — score: {s:.4f}")
        print(f"   {info}\n")


#### Aquí creamos la interfaz de búsqueda con un bucle simple

In [42]:
while True:
    q = input("\nEscribe tu búsqueda (o 'salir'): ").strip()
    if q.lower() == "salir":
        print("Cerrando buscador...")
        break
    
    top_idx, scores = buscar(q, k=10)
    
    if len(top_idx) == 0:
        print("No se encontraron resultados.")
    else:
        mostrar_resultados(top_idx, scores)



Escribe tu búsqueda (o 'salir'):  action adventure



RESULTADOS:
1. Action Point — score: 5.5188
   D.C. is the crackpot owner of Action Point -- a low-rent, out-of-control amusement park where the rides are designed with minimum safety for maximum fun. Just as his estranged daughter Boogie comes to visit, a corporate mega-park opens nearby and jeopardizes the future of Action Point. To save his beloved park and his relationship with Boogie, D.C. and his loony crew of misfits must risk everything to pull out all the stops and save the day.

2. Action Jackson — score: 4.4142
   Due to the use of excessive force in arresting a sexual predator, Detective Lt. Jericho "Action" Jackson (Carl Weathers) is demoted to police sergeant. What's worse, the father of the man he roughed up, Detroit industrialist Peter Dellaplane (Craig T. Nelson), declares war on Jackson, framing him for the murder of his own wife, Patrice (Sharon Stone). But Jericho manages to elude arrest and, with the help of a drug-addicted singer named Sydney (Vanity), goes after


Escribe tu búsqueda (o 'salir'):  salir


Cerrando buscador...


### Interpretación de resultados

##### El sistema, sí funciona correctamente para responder consultas relacionadas con películas. Las búsquedas nos dan los resultados coherentes lo que significa que el preprocesamiento limpió bien los textos, eliminó ruido y pudimos quedarnos solo con información útil de las pelis. Las consultas de prueba como “action adventure” nos dan títulos que sí tienen relación con los géneros, sinopsis o temas mencionados.

##### El detalle más importante y diferente a lo que hicimos en clase es que el dataset tenía más de 17 mil películas y un vocabulario de más de 108 mil palabras únicas. Cuando intenté construir la matriz TF-IDF completa, el sistema intentó usar más de 14 GB de memoria, lo cual no es posible en mi computadora que ya está muriendo. Por eso tuve que cambiar la representación y usar una estructura dispersa (sparse) que igual ocupé en el proyecto aunque luego ya en la computadora de mi compañero si funconó el normal, donde solo guardo las palabras que realmente aparecen en cada documento. Esto permitió continuar sin problemas y el sistema pudo calcular los pesos TF-IDF sin colapsar la memoria de la compu.

##### Usando similaridad x coseno, se ordenan las películas según qué tan relevantes son para cada consulta, y los resultados están bien de acuerdo al contenido. Aunque el sistema es básico, cumple bien con lo que se pedía para el examen. Se pueden mejorar varias cosas: usar lematización y sinónimos en el preprocesamiento, ajustar TF-IDF o probar BM25 y embeddings, evaluar con métricas como precisión y recall, y optimizar memoria usando matrices dispersas o ajustando la dimensionalidad.